# Baseline Model — Gurgaon House Price Prediction
**Improvements:** removed duplicate import, added RMSE + test R², train vs test R² overfitting check, feature importance plot.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
# FIX: removed duplicate ColumnTransformer import that appeared twice originally

In [ ]:
df = pd.read_csv('gurgaon_properties_post_feature_selection.csv')
print(df.shape)
df.head()

In [ ]:
X = df.drop(columns=['price'])
y = df['price']

# Log-transform target to reduce skewness
y_transformed = np.log1p(y)

columns_to_encode = ['property_type','sector','balcony','agePossession',
                     'furnishing_type','luxury_category','floor_category']
num_cols = [c for c in X.columns if c not in columns_to_encode]

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),  num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), columns_to_encode),
], remainder='passthrough')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

## Cross-Validation (10-Fold)

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

print(f"CV R2  : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Train / Test Evaluation
Added RMSE and train R² to detect overfitting (was missing in original).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)

y_pred_log   = pipeline.predict(X_test)
y_pred       = np.expm1(y_pred_log)
y_test_orig  = np.expm1(y_test)

y_train_pred = np.expm1(pipeline.predict(X_train))
y_train_orig = np.expm1(y_train)

mae   = mean_absolute_error(y_test_orig, y_pred)
rmse  = np.sqrt(mean_squared_error(y_test_orig, y_pred))

# R² on original scale
ss_res_test  = np.sum((y_test_orig  - y_pred)**2)
ss_tot_test  = np.sum((y_test_orig  - y_test_orig.mean())**2)
test_r2      = 1 - ss_res_test / ss_tot_test

ss_res_train = np.sum((y_train_orig - y_train_pred)**2)
ss_tot_train = np.sum((y_train_orig - y_train_orig.mean())**2)
train_r2     = 1 - ss_res_train / ss_tot_train

print(f"Train R2 : {train_r2:.4f}")
print(f"Test  R2 : {test_r2:.4f}")
print(f"Test  MAE : {mae:.4f} Cr")
print(f"Test  RMSE: {rmse:.4f} Cr")
if train_r2 - test_r2 > 0.05:
    print("WARNING: Possible overfitting (Train R2 >> Test R2)")

## Feature Importance
Was completely missing in the original notebook.

In [ ]:
rf_model = pipeline.named_steps['regressor']
ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
ohe_feature_names = ohe.get_feature_names_out(columns_to_encode).tolist()
all_feature_names = num_cols + ohe_feature_names

importances = rf_model.feature_importances_
fi_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(9, 6))
plt.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='#6366f1')
plt.xlabel('Feature Importance')
plt.title('Top 20 Feature Importances — Random Forest Baseline')
plt.tight_layout()
plt.show()

## Residual Analysis
Was missing in the original notebook. Helps identify systematic errors.

In [ ]:
residuals = y_test_orig.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred, residuals, alpha=0.4, color='#6366f1')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Price (Cr)')
axes[0].set_ylabel('Residual (Cr)')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=40, color='#6366f1', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (Cr)')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()
print(f"Residual mean : {residuals.mean():.4f} (should be ~0)")
print(f"Residual std  : {residuals.std():.4f}")